In [1]:
import sys
import os
# sys.path.append(os.path.join(os.path.dirname(__file__), '~/ws_python12_ZED/samples/depth sensing/depth sensing/python/ogl_viewer'))
target_dir = os.path.join(os.path.expanduser('~'), 'ws_python12_ZED','samples','depth sensing','depth sensing','python','ogl_viewer')
if target_dir not in sys.path:
    sys.path.append(target_dir)
import viewer as gl
# import ogl_viewer.viewer as gl

import pyzed.sl as sl
import argparse

mem_type = sl.MEM.GPU
init = sl.InitParameters(depth_mode=sl.DEPTH_MODE.NEURAL,
                         coordinate_units=sl.UNIT.METER,
                         coordinate_system=sl.COORDINATE_SYSTEM.RIGHT_HANDED_Y_UP)
zed = sl.Camera()
status = zed.open(init)
res = sl.Resolution()
res.width = -1
res.height = -1

# Get the first PC to retrieve the resolution
point_cloud = sl.Mat()
zed.retrieve_measure(point_cloud, sl.MEASURE.XYZRGBA, mem_type, res)
res = point_cloud.get_resolution()



[2026-05-10 20:35:13 UTC][ZED][INFO] Logging level INFO
[2026-05-10 20:35:14 UTC][ZED][INFO] Using USB input... Switched to default resolution HD720
[2026-05-10 20:35:14 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-05-10 20:35:14 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-05-10 20:35:14 UTC][ZED][INFO] [Init]  Video mode: HD720@30
[2026-05-10 20:35:14 UTC][ZED][INFO] [Init]  Serial Number: S/N 38514689
[2026-05-10 20:35:14 UTC][ZED][INFO] [Init]  Depth mode selected: NEURAL. Ensure this mode matches your application's performance and accuracy requirements. See https://www.stereolabs.com/docs/depth-sensing/depth-modes for help.
[2026-05-10 20:35:16 UTC][ZED][WARNING] [Init]  Self-calibration was skipped due to low texture or occlusion in the scene. Tracking accuracy may be affected.


In [4]:
if zed.grab() <= sl.ERROR_CODE.SUCCESS:
    zed.retrieve_measure(point_cloud, sl.MEASURE.XYZRGBA, mem_type, res)
else:
    print('eee')

#point_cloud_cpu = sl.Mat()
# 3. GPU 메모리 데이터를 CPU로 복사
# update_cpu_from_gpu()를 사용하면 간편하게 동기화됩니다.
point_cloud.update_cpu_from_gpu() 
# 또는 복사본을 만들려면: point_cloud_gpu.copy_to(point_cloud_cpu)
#point_cloud.get_data()

SUCCESS

In [5]:
import open3d as o3d
import numpy as np

# 1. 먼저 sl.Mat 데이터를 넘파이 배열로 가져옵니다.
# (이미 update_cpu_from_gpu()가 호출되었다고 가정합니다)
point_cloud_np = point_cloud.get_data()

# 2. NaN 값을 제거합니다 (Open3D는 NaN이 있으면 에러가 나거나 안 보일 수 있습니다)
# 3차원(256, 448, 4)을 2차원(N, 4)으로 펼치기
points = point_cloud_np.reshape(-1, 4)
# X, Y, Z 중 하나라도 NaN이 아닌 데이터만 추출
valid_points = points[~np.isnan(points[:, :3]).any(axis=1)]

# 3. Open3D 포인트 클라우드 객체 생성
pcd = o3d.geometry.PointCloud()

# 4. 이제 넘파이 배열의 XYZ 값만 넣어줍니다.
pcd.points = o3d.utility.Vector3dVector(valid_points[:, :3])

# 5. 시각화 실행
o3d.visualization.draw_geometries([pcd])


In [6]:
zed.close()